<a href="https://colab.research.google.com/github/bearsmithnu/bearsmithnu/blob/main/retrieval-models-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Retrieval Models and Evaluation



In this notebook, you will evaluate results ranking on a test collection. First, you'll compute the mean average precision of a baseline BM25 model. Then you'll implement a query-likelihood retrieval model based on the same inverted index.

This notebook uses the [Pyserini](http://pyserini.io/) library, a Python interface to [Anserini](http://anserini.io) and thus to [Lucene](https://lucene.apache.org/), a widely-used open-source search engine. This library is written and maintained by Jimmy Lin and his colleagues at the University of Waterloo.


We start by installing the python interface. Since it calls the underlying Lucene search engine, which is written in Java, we make sure we point to an appropriate Java installation. If like Colab you don't have Java 21, uncomment the following code and run it, or whatever makes sense for your platform.

In [ ]:
## Uncomment the following code to install Java 21 on Colab
!apt-get install openjdk-21-jre-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java
!java -version

In [ ]:
!pip install pyserini
# You can change this to gpu if you have one.
# It's a pyserini dependency, but we won't need it until the next assignment.
!pip install faiss-cpu

You can use the `LuceneSearcher` to search over an index. We can initialize the searcher with a pre-built index, which Pyserini will automatically download if it hasn't already. Note that the index takes up 1.6GB of disk.

In [ ]:
from pyserini.search.lucene import LuceneSearcher

searcher = LuceneSearcher.from_prebuilt_index('robust04')

Now we can search for a query and inspect the results:

In [ ]:
hits = searcher.search('black bear attacks', 1000)

# Prints the first 10 hits
for i in range(0, 10):
    print(f'{i+1:2} {hits[i].docid:15} {hits[i].score:.5f}')

The `IndexReaderUtils` class provides various methods to read the index directly. For example, we can fetch a raw document from the index given its `docid`:

In [ ]:
from pyserini.index import LuceneIndexReader
from IPython.core.display import display, HTML

reader = LuceneIndexReader.from_prebuilt_index('robust04')

doc = reader.doc('LA092790-0015').raw()
display(HTML('<div style="font-family: Times New Roman; padding-bottom:10px">' + doc + '</div>'))

Note that the result is exactly the same as displaying the hit contents above. Given the raw text, we can obtain its analyzed form (i.e., tokenized, stemmed, stopwords removed, etc.). Here we show the first ten tokens:

In [ ]:
analyzed = reader.analyze(doc)
analyzed[0:10]

The index also stores the raw document vector, which we can obtain as a Python dictionary of analyzed terms to counts (i.e., term frequency).
For brevity, we only look at terms that appear more than once:

In [ ]:
doc_vector = reader.get_document_vector('LA092790-0015')
{ k: v for (k, v) in doc_vector.items() if v >1 }

## Evaluating Ranked Results

We can load some standard evaluation sets such as Robust04, which contains 250 queries, or "topics" as the TREC conferences call them.

In [ ]:
from pyserini.search import get_topics
topics = get_topics('robust04')
print(f'{len(topics)} queries total')

The topics are in a dictionary, whose keys are integers uniquely identifying each query. Each topic contains the following fields:

* `title`: TREC's term for the brief query a user might actually type;
* `description`: a longer form of the query in the form of a complete sentence; and
* `narrative`: a description of what the user is looking for and what kinds of results would be relevant or non-relevant.

In [ ]:
topics[301]

For the purpose of your experiments, we'll divide them into a development and test set.

In [ ]:
dev_topics = {k:topics[k] for k in list(topics.keys())[:125]}
test_topics = {k:topics[k] for k in list(topics.keys())[125:]}

Now, we'll fetch the relevance judgments for the Robust04 queries, which TREC calls "qrels".

In [ ]:
from urllib.request import urlopen

qfile = 'https://github.com/castorini/anserini-tools/blob/63ceeab1dd94c1221f29b931d868e8fab67cc25c/topics-and-qrels/qrels.robust04.txt?raw=true'
qrels = []
for line in urlopen(qfile):
  qid, round, docid, score = line.strip().split()
  qrels.append([int(qid), 0, docid.decode('UTF-8'), int(score)])
#qrels = [line.strip().split() for line in urlopen(qfile)]

Each record in the qrel contains four fields:

1. the numeric identifier of the query;
2. the round of relevance feedback, which is here always 0;
3. the identifier of a documennt that has been judged; and
4. the relevance score of that document.

In Robust04, all relevance judgments are binary, i.e., 1 or 0. Note that not all non-relevant documents are recorded. The qrel file only contains those documents the annotators actually looked at; the vast majority of documents in the collection have not been judged. In IR evaluation, we assume that unannotated documents are non-relevant.

In [63]:
qrels[0:10]

[[301, 0, 'FBIS3-10082', 1],
 [301, 0, 'FBIS3-10169', 0],
 [301, 0, 'FBIS3-10243', 1],
 [301, 0, 'FBIS3-10319', 0],
 [301, 0, 'FBIS3-10397', 1],
 [301, 0, 'FBIS3-10491', 1],
 [301, 0, 'FBIS3-10555', 0],
 [301, 0, 'FBIS3-10622', 1],
 [301, 0, 'FBIS3-10634', 0],
 [301, 0, 'FBIS3-10635', 0]]

## Computing Mean Average Precision

The Robust04 collection uses binary relevance judgments and usually has multiple relevant results for each query. It is thus common to use **mean average precision** (MAP) to evaluate retrieval performance on it. Remember from class that MAP adds the precision at the position of each _relevant_ document in a ranked list and then divides by the total number of relevant documents. So that we don't have to scan through the entire collection, we usually evaluate MAP at some maximum rank value, such as 100 or 1000. We simply stop scanning at that maximum rank.

As we saw above, you should pass a query string (the `title` of a topic) and the desired number of results to the `search` method of the `searcher` object.

In [84]:
hits = searcher.search(dev_topics[355]['title'], 1000)
[(hit.docid, hit.score) for hit in hits[0:10]]

[('FBIS4-20436', 11.349300384521484),
 ('FBIS3-23683', 9.126500129699707),
 ('FBIS3-21238', 8.309300422668457),
 ('FBIS4-44915', 7.935699939727783),
 ('FBIS4-20602', 7.6006999015808105),
 ('FBIS4-47382', 7.529600143432617),
 ('FT943-1589', 7.480100154876709),
 ('LA071789-0059', 7.451399803161621),
 ('FBIS4-22145', 7.215099811553955),
 ('FBIS4-44667', 7.106500148773193)]

For this assignment, evaluate MAP@1000 for the list of `test_topics` we created above. You should process the `qrels` data to find the relevant results for each query.

In [95]:
## TODO [35 points]: Compute MAP@1000 for test_topics

# Reformat the relevance judgements so that they're easier to query.
relevances = dict()
for qrel in qrels:
  # Each key is composed of a (topic, document) and its value is the relevance.
  relevances[(qrel[0], qrel[2])] = qrel[3]

# Begin counting the sum of average precision across queries, and the number
# of queries computed.
avg_precision_sum = 0
avg_precision_queries = 0

# Iterate through each topic in the test set and calcualte its AP:
for topic_key, topic in test_topics.items():
  # Define an accumulator for precision and accumulators for the number of
  # relevant, and total, results.
  precision_sum = 0
  rel_results = 0
  num_results = 0

  # Get the top 1000 results for the topic.
  for result in searcher.search(topic["title"], 1000):
    num_results += 1
    # Check if the result is relevant.
    if relevances.get((topic_key, result.docid)) == 1:
      # If so, find the precision@ its position.
      rel_results += 1
      precision_sum += rel_results / num_results

  # Calculate the average precision for this topic and add it to the
  # accumulator.
  if (rel_results > 0):
    avg_precision_sum += precision_sum / rel_results
  avg_precision_queries += 1

# Calculate mean average precision @ 1000 for the entire test set.
map_at_1000 = avg_precision_sum / avg_precision_queries


In [96]:
map_at_1000

0.30002159372138726

## Implementing a Query Likelihood Model

The default `LuceneSearcher` in pyserini uses a BM25 model. In this part of the assignment, you'll interact directly with the index to implement a query likelihood retrieval model.  For more details on pyserini's index API, [see the documentation](https://github.com/castorini/pyserini/blob/master/docs/usage-indexreader.md).

We've already created an `IndexReader` for the Robust04 collection and assigned it to `reader`. Remember that we can pass documents or queries through the index's predefined analysis pipeline, a series of operations like stemming, stopword removal, etc.

In [ ]:
reader.analyze('Are there black bear attacks there?')

For most retrieval models, including BM25 and query likelihood, we'll want to know the overall statistics for the terms we retrieve from the index. By default, pyserini will do stemming and other "analysis" steps on the terms you give it. But since we're already working with stemmed queries, we'll tell it not to perform this analysis now.

In [ ]:
# Just checking that our term is already in its index form.
print(reader.analyze('bear'))

df, cf = reader.get_term_counts('bear', analyzer=None)
print(f'The document frequency is {df} and the collection frequency is {cf}.')

You might also want to know summary statistics about the collection.

In [ ]:
reader.stats()

Now let's traverse the postings for a term.

In [ ]:
posting_list = reader.get_postings_list('bear', analyzer=None)
print(f'There are {len(posting_list)} postings.')

posting = posting_list[100]
print(f'Each posting contains a document ID ({posting.docid}), term frequency ({posting.tf}), and position list ({posting.positions}).')

print(f'searcher has a method for turning internal integer IDs like {posting.docid} into external IDs like {searcher.doc(posting.docid).docid()}.')

If, instead of mapping terms to documents, you want to use a document name to get term information, see above for the `get_document_vector` method.

Finally, it will be useful to have some information about each document in the collection, which is saved in a file in the GitHub repository.

In [ ]:
import json
docfile = urlopen('https://github.com/dasmiq/cs6200-retrieval-models/raw/refs/heads/main/doclen.jsonl')
docinfo = [json.loads(line) for line in docfile]

Each record in that file contains the document `id`, its interned numeric `iid`, and the number of tokens `n`.

In [ ]:
docinfo[0]

Use Dirichlet smoothing with $\mu$ equal to the mean document length in the collection.

In [ ]:
 # TODO [2 points]: Compute the mean document length from collection statistics.
def get_dirichlet_mu(collection):
  doc_tokens_sum = 0
  doc_count_sum = 0
  for doc in collection:
    doc_tokens_sum += doc["n"]
    doc_count_sum += 1
  return doc_tokens_sum / doc_count_sum

get_dirichlet_mu(docinfo)

Now you should have all the information you need to implement a query likelihood retrieval model.

In [ ]:
## TODO [50 points]: Implement a query likelihood model with Dirichlet smoothing.
## For a given query, index reader, and constant k, return the k documents with the highest query likelihoods.
## Think about what data structure would let you keep the top k.
def query_likelihood(query, in_reader, k):
  reader_stats = in_reader.stats()

  # Find the size of the corpus.
  corpus_size = reader_stats["total_terms"]

  # Find the average document length.
  mu = reader_stats["total_terms"] / reader_stats["documents"]

  # Find the frequency of each query term within the entire corpus.
  term_freqs_in_corpus = {}
  for term in in_reader.analyze(query):
    print(term)
    doc_freq, col_freq = in_reader.get_term_counts(term, analyzer=None)
    term_freqs_in_corpus[term] = col_freq
    print(col_freq)


  # For each document:

    # For each query term:
      # If it has no chance of entering the top K documents,
      # skip the rest of this document.
      # Calculate the probability of the query term within the document.
    # Find the product of all the query term likelihoods and, if this document
    # is in the top K, store it.

  # Return the top K documents.
  return


Now you can reuse some of your evaluation code for mean average precision from above.

In [ ]:
## TODO [13 points]: Compute the MAP@1000 of your query likelihood model on the test_topics set.